To run the notebook, first run on the command line

In [8]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries and Setup

In [9]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
import ultralytics
import random

# Check Ultralytics version
print(f"Ultralytics version: {ultralytics.__version__}")
if ultralytics.__version__ < '8.3.146':
    print("Warning: Ultralytics version is outdated. Please update with 'pip install -U ultralytics'")

# Set random seed for reproducibility
random.seed(42)

Ultralytics version: 8.3.146


## 2. Define YOLO Annotation Function

In [10]:
def create_yolo_annotations(root_dir, output_dir, partition='train'):
    """
    Convert annotations.json to YOLO format for a given partition (train/val/test).
    Creates .txt files with class_id and normalized bbox coordinates.
    Returns list of image paths for the partition.
    """
    # Load annotations
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return [], {}
    anns = json.load(open(anns_file))
    categories = {c['id']: c['name'] for c in anns['categories']}
    image_info = {img['id']: img for img in anns['images']}

    # Check if partition exists
    if partition not in anns['splits']['chessred2k']:
        print(f"Error: Partition '{partition}' not found in annotations.json")
        return [], categories

    # Get split IDs
    split_ids = np.asarray(anns['splits']['chessred2k'][partition]['image_ids']).astype(int)
    image_ids = [img['id'] for img in anns['images'] if img['id'] in split_ids]
    image_paths = []
    images_processed = 0
    images_skipped = 0

    # Create output directory for labels
    label_dir = os.path.join(output_dir, partition, 'labels')
    image_dir = os.path.join(output_dir, partition, 'images')
    os.makedirs(label_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    for image_id in image_ids:
        img_info = image_info[image_id]
        file_name = img_info['path']
        img_path = os.path.join(root_dir, file_name)
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            images_skipped += 1
            continue
        width, height = img_info['width'], img_info['height']

        # Copy image to output directory
        output_img_path = os.path.join(image_dir, os.path.basename(file_name))
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not load image {img_path}")
            images_skipped += 1
            continue
        cv2.imwrite(output_img_path, img)
        image_paths.append(os.path.join(partition, 'images', os.path.basename(file_name)))
        images_processed += 1

        # Create YOLO annotation file
        label_path = os.path.join(label_dir, os.path.splitext(os.path.basename(file_name))[0] + '.txt')
        with open(label_path, 'w') as f:
            for piece in anns['annotations']['pieces']:
                if piece['image_id'] == image_id and 'bbox' in piece:
                    x, y, w, h = piece['bbox']
                    class_id = piece['category_id']
                    # Normalize coordinates: center_x, center_y, width, height
                    center_x = (x + w / 2) / width
                    center_y = (y + h / 2) / height
                    norm_w = w / width
                    norm_h = h / height
                    # Ensure coordinates are within [0, 1]
                    if 0 <= center_x <= 1 and 0 <= center_y <= 1 and norm_w > 0 and norm_h > 0:
                        f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                    else:
                        print(f"Warning: Invalid bbox for image_id {image_id}: {piece['bbox']}")

    print(f"Processed {images_processed} images for {partition} split, skipped {images_skipped}")
    return image_paths, categories

## 3. Define Data YAML Creation Function

In [11]:
def create_data_yaml(root_dir, output_dir, train_paths, val_paths, test_paths, categories):
    """
    Create data.yaml file for YOLOv11 training with absolute paths.
    """
    # Get absolute paths
    train_dir = os.path.abspath(os.path.join(output_dir, 'train', 'images'))
    val_dir = os.path.abspath(os.path.join(output_dir, 'val', 'images'))
    test_dir = os.path.abspath(os.path.join(output_dir, 'test', 'images'))

    yaml_content = f"""
train: {train_dir}
val: {val_dir}
test: {test_dir}
nc: {len(categories)}
names: {list(categories.values())}
"""
    yaml_path = os.path.join(output_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    # Print data.yaml contents for debugging
    print(f"data.yaml contents:\n{yaml_content}")
    return yaml_path

## 4. Training Pipeline

In [12]:
root_dir = ''  # Update with your dataset path
output_dir = 'yolo_chess_dataset'  # Output directory for YOLO format
model_name = 'yolo11n.pt'  # Pretrained YOLOv11 model
img_size = 640  # Training image size
epochs = 50  # Number of training epochs
batch_size = 16  # Batch size (adjust based on GPU memory)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create YOLO annotations
print("Converting annotations for training set...")
train_paths, categories = create_yolo_annotations(root_dir, output_dir, 'train')
print("Converting annotations for validation set...")
val_paths, _ = create_yolo_annotations(root_dir, output_dir, 'val')
print("Converting annotations for test set...")
test_paths, _ = create_yolo_annotations(root_dir, output_dir, 'test')

Converting annotations for training set...
Processed 1442 images for train split, skipped 0
Converting annotations for validation set...
Processed 330 images for val split, skipped 0
Converting annotations for test set...
Processed 306 images for test split, skipped 0


In [13]:

# Debug: Check available splits
anns_file = os.path.join(root_dir, 'annotations.json')
if os.path.exists(anns_file):
    anns = json.load(open(anns_file))
    print(f"Available splits in annotations.json: {list(anns['splits']['chessred2k'].keys())}")
else:
    print(f"Error: Annotations file {anns_file} not found")

# Check for empty splits
if not train_paths:
    print("Error: No training images found. Cannot proceed with training.")
elif not val_paths:
    print("Warning: No validation images found. Using training set for validation.")
    val_paths = train_paths  # Fallback to train set for validation

# Create data.yaml
yaml_path = create_data_yaml(root_dir, output_dir, train_paths, val_paths, test_paths, categories)
print(f"Created data.yaml at {yaml_path}")

# Verify directories
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(output_dir, split, 'images')
    if os.path.exists(img_dir) and len(os.listdir(img_dir)) > 0:
        print(f"{split} images directory: {img_dir} contains {len(os.listdir(img_dir))} images")
    else:
        print(f"Warning: {split} images directory {img_dir} is empty or does not exist")

# Load and train YOLOv11 model
print(f"Loading pretrained model {model_name} on {device}")
model = YOLO(model_name)
print("Starting training...")
results = model.train(
    data=yaml_path,
    imgsz=img_size,
    epochs=epochs,
    batch=batch_size,
    name='chess_piece_detection',
    plots=True,
    device=device,
    patience=20,  # Early stopping after 20 epochs without improvement
    save=True,  # Save checkpoints
    save_period=10  # Save every 10 epochs
)

Available splits in annotations.json: ['train', 'val', 'test']
data.yaml contents:

train: d:\VCOM para entregar\yolo_chess_dataset\train\images
val: d:\VCOM para entregar\yolo_chess_dataset\val\images
test: d:\VCOM para entregar\yolo_chess_dataset\test\images
nc: 13
names: ['white-pawn', 'white-rook', 'white-knight', 'white-bishop', 'white-queen', 'white-king', 'black-pawn', 'black-rook', 'black-knight', 'black-bishop', 'black-queen', 'black-king', 'empty']

Created data.yaml at yolo_chess_dataset\data.yaml
train images directory: yolo_chess_dataset\train\images contains 1442 images
val images directory: yolo_chess_dataset\val\images contains 330 images
test images directory: yolo_chess_dataset\test\images contains 306 images
Loading pretrained model yolo11n.pt on cuda
Starting training...
New https://pypi.org/project/ultralytics/8.3.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.146  Python-3.11.5 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GP

train: Scanning D:\VCOM para entregar\yolo_chess_dataset\train\labels.cache... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:00<?, ?it/s]


val: Fast image access  (ping: 0.10.1 ms, read: 339.640.2 MB/s, size: 1870.3 KB)


val: Scanning D:\VCOM para entregar\yolo_chess_dataset\val\labels.cache... 330 images, 0 backgrounds, 0 corrupt: 100%|██████████| 330/330 [00:00<?, ?it/s]


Plotting labels to runs\detect\chess_piece_detection3\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000588, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\detect\chess_piece_detection3
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      50/50       2.3G     0.2927     0.2567     0.7743         37        640: 100%|██████████| 91/91 [00:47<00:00,  1.92it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:09<00:00,  1.13it/s]

                   all        330       6132      0.996      0.996      0.995       0.85



50 epochs completed in 0.440 hours.
Optimizer stripped from runs\detect\chess_piece_detection3\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\chess_piece_detection3\weights\best.pt, 5.5MB

Validating runs\detect\chess_piece_detection3\weights\best.pt...
Ultralytics 8.3.146  Python-3.11.5 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLO11n summary (fused): 100 layers, 2,584,687 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:08<00:00,  1.26it/s]


                   all        330       6132      0.993      0.992      0.995      0.871
            white-pawn        330       1625      0.999      0.995      0.995      0.848
            white-rook        281        447      0.996          1      0.995      0.848
          white-knight        220        274          1      0.989      0.995      0.866
          white-bishop        235        335          1      0.991      0.995      0.864
           white-queen        126        126      0.992          1      0.995      0.915
            white-king        330        330      0.997      0.998      0.995      0.912
            black-pawn        330       1511      0.998      0.997      0.995      0.856
            black-rook        280        471      0.998      0.993      0.995      0.883
          black-knight        125        178      0.983      0.972      0.994      0.819
          black-bishop        223        380      0.998      0.979      0.995      0.858
           black-quee

## 5. Test Model on Sample Image

In [21]:
test_image_path = os.path.join(output_dir, test_paths[0])
if not os.path.exists(test_image_path):
    print(f"Error: Test image {test_image_path} not found")
else:
    print("Testing model on a sample image...")
    results = model.predict(test_image_path, save=False, conf=0.5)

    # Load and display image with predictions
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"Error: Could not load test image {test_image_path}")
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()
                cls = int(box.cls.item())
                label = f"{categories[cls]} {conf:.2f}"
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, label, (x1, max(y1 - 10, 10)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        plt.imshow(img)
        plt.axis('off')
        plt.title(f"YOLOv11 Predictions on {os.path.basename(test_image_path)}")
        plt.show()

        print(f"Test image: {test_image_path}")
        print(f"Detected {len(result.boxes)} objects:")
        for box in result.boxes:
            cls = int(box.cls.item())
            conf = box.conf.item()
            print(f" - {categories[cls]} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

Testing model on a sample image...

image 1/1 d:\VCOM para entregar\yolo_chess_dataset\test\images\G000_IMG000.jpg: 640x640 8 white-pawns, 2 white-rooks, 2 white-knights, 2 white-bishops, 1 white-queen, 1 white-king, 8 black-pawns, 2 black-rooks, 2 black-knights, 2 black-bishops, 1 black-queen, 1 black-king, 57.7ms
Speed: 7.2ms preprocess, 57.7ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 640x480 with 1 Axes>

Test image: yolo_chess_dataset\test\images\G000_IMG000.jpg
Detected 32 objects:
 - white-rook at [2410.435546875, 1391.2392578125, 2556.31640625, 1585.6058349609375] with confidence 0.98
 - black-rook at [524.1851196289062, 963.0350341796875, 653.1126098632812, 1145.84228515625] with confidence 0.98
 - white-queen at [1670.716796875, 1653.9818115234375, 1824.3785400390625, 1938.3594970703125] with confidence 0.97
 - black-pawn at [1477.205322265625, 801.9133911132812, 1569.2320556640625, 952.1345825195312] with confidence 0.97
 - white-king at [1856.922607421875, 1540.57421875, 2021.90234375, 1849.4783935546875] with confidence 0.97
 - black-knight at [1538.9376220703125, 599.3533325195312, 1642.3076171875, 793.9320068359375] with confidence 0.97
 - black-queen at [1055.0556640625, 711.1854858398438, 1183.136474609375, 966.1668701171875] with confidence 0.97
 - black-rook at [1690.124755859375, 567.7242431640625, 1796.2041015625, 737.964599609375] with confidence 0.97
 - white-knight a

## 6. Inference (Utility Functions)

In [22]:
def load_categories(root_dir):
    """
    Load category names from annotations.json.
    """
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return {}
    with open(anns_file, 'r') as f:
        anns = json.load(f)
    return {c['id']: c['name'] for c in anns['categories']}

def get_test_image_paths(output_dir):
    """
    Get list of image paths in the test set.
    """
    test_image_dir = os.path.join(output_dir, 'test', 'images')
    if not os.path.exists(test_image_dir):
        print(f"Error: Test images directory {test_image_dir} not found")
        return []
    return [os.path.join(test_image_dir, img) for img in os.listdir(test_image_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

def test_on_image(model, image_path, categories, output_dir, conf_threshold=0.5):
    """
    Run inference on a single image, display results, and save annotated image.
    """
    if not os.path.exists(image_path):
        print(f"Error: Image {image_path} not found")
        return

    results = model.predict(image_path, save=False, conf=conf_threshold)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image {image_path}")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_annotated = img.copy()

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"YOLOv11 Predictions on {os.path.basename(image_path)}")
    plt.show()

    predictions_dir = os.path.join(output_dir, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)
    output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(image_path)}")
    cv2.imwrite(output_image_path, img_annotated)
    print(f"Saved annotated image to: {output_image_path}")

    print(f"Test image: {image_path}")
    print(f"Detected {len(result.boxes)} objects:")
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = box.conf.item()
        print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

## 7. Run Inference on Test Image

In [23]:
root_dir = ''  # Update with your dataset path
output_dir = 'yolo_chess_dataset'
model_path = 'runs/detect/chess_piece_detection/weights/best.pt'
conf_threshold = 0.7

# Load categories and test images
categories = load_categories(root_dir)
test_image_paths = get_test_image_paths(output_dir)


# Limit to 3 test images or fewer if available
test_image_paths = test_image_paths[:3]
print(f"Loading trained model from {model_path}")
model = YOLO(model_path)

for test_image_path in test_image_paths:
    if not os.path.exists(test_image_path):
        print(f"Error: Image {test_image_path} not found")
        continue

    print(f"Testing on image: {test_image_path}")
    
    # Run inference
    results = model.predict(test_image_path, save=False, conf=conf_threshold)
    
    # Load original image
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"Error: Could not load image {test_image_path}")
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_annotated = img.copy()  # For annotations (BGR)

    # Draw bounding boxes and larger labels
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            # Draw rectangle (BGR for OpenCV, red)
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            # Add larger label
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    # Convert annotated image to RGB for display
    img_annotated_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
    
    # Display side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(img_rgb)
    ax1.set_title("Original Image")
    ax1.axis('off')
    ax2.imshow(img_annotated_rgb)
    ax2.set_title(f"Predictions on {os.path.basename(test_image_path)}")
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

    # Save annotated image
    predictions_dir = os.path.join(output_dir, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)
    output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(test_image_path)}")
    cv2.imwrite(output_image_path, img_annotated)
    print(f"Saved annotated image to: {output_image_path}")

    # Print results
    print(f"Test image: {test_image_path}")
    print(f"Detected {len(result.boxes)} objects:")
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = box.conf.item()
        print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

Loading trained model from runs/detect/chess_piece_detection/weights/best.pt
Testing on image: yolo_chess_dataset\test\images\G000_IMG000.jpg

image 1/1 d:\VCOM para entregar\yolo_chess_dataset\test\images\G000_IMG000.jpg: 640x640 8 white-pawns, 2 white-rooks, 2 white-knights, 2 white-bishops, 1 white-queen, 1 white-king, 8 black-pawns, 2 black-rooks, 2 black-knights, 2 black-bishops, 1 black-queen, 1 black-king, 57.8ms
Speed: 6.7ms preprocess, 57.8ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1200x600 with 2 Axes>

Saved annotated image to: yolo_chess_dataset\predictions\pred_G000_IMG000.jpg
Test image: yolo_chess_dataset\test\images\G000_IMG000.jpg
Detected 32 objects:
 - white-rook at [2410.435546875, 1391.2392578125, 2556.31640625, 1585.6058349609375] with confidence 0.98
 - black-rook at [524.1851196289062, 963.0350341796875, 653.1126098632812, 1145.84228515625] with confidence 0.98
 - white-queen at [1670.716796875, 1653.9818115234375, 1824.3785400390625, 1938.3594970703125] with confidence 0.97
 - black-pawn at [1477.205322265625, 801.9133911132812, 1569.2320556640625, 952.1345825195312] with confidence 0.97
 - white-king at [1856.922607421875, 1540.57421875, 2021.90234375, 1849.4783935546875] with confidence 0.97
 - black-knight at [1538.9376220703125, 599.3533325195312, 1642.3076171875, 793.9320068359375] with confidence 0.97
 - black-queen at [1055.0556640625, 711.1854858398438, 1183.136474609375, 966.1668701171875] with confidence 0.97
 - black-rook at [1690.124755859375, 567.7242431640

<Figure size 1200x600 with 2 Axes>

Saved annotated image to: yolo_chess_dataset\predictions\pred_G000_IMG001.jpg
Test image: yolo_chess_dataset\test\images\G000_IMG001.jpg
Detected 31 objects:
 - black-bishop at [800.528076171875, 1014.590087890625, 928.5126342773438, 1245.7861328125] with confidence 0.99
 - white-pawn at [1951.18603515625, 1862.951416015625, 2087.390625, 2075.510009765625] with confidence 0.98
 - white-pawn at [1976.6031494140625, 1536.7965087890625, 2098.899658203125, 1731.3743896484375] with confidence 0.98
 - black-rook at [894.6829223632812, 827.1732788085938, 1011.7743530273438, 1014.5426025390625] with confidence 0.98
 - black-pawn at [742.946044921875, 1788.1422119140625, 874.5530395507812, 1990.0491943359375] with confidence 0.97
 - black-pawn at [926.9281005859375, 1323.9677734375, 1041.5830078125, 1508.5] with confidence 0.97
 - black-pawn at [1058.77490234375, 961.4473266601562, 1160.989990234375, 1137.93896484375] with confidence 0.97
 - black-pawn at [1017.8687133789062, 1076.44140625, 112

<Figure size 1200x600 with 2 Axes>

Saved annotated image to: yolo_chess_dataset\predictions\pred_G000_IMG002.jpg
Test image: yolo_chess_dataset\test\images\G000_IMG002.jpg
Detected 32 objects:
 - black-bishop at [2231.82568359375, 1226.13720703125, 2363.6123046875, 1468.00048828125] with confidence 0.98
 - black-rook at [1686.5965576171875, 741.5860595703125, 1800.1351318359375, 925.5679321289062] with confidence 0.98
 - white-pawn at [907.6978149414062, 1436.2520751953125, 1030.031982421875, 1626.7333984375] with confidence 0.98
 - black-queen at [1991.2928466796875, 955.4883422851562, 2135.970458984375, 1242.0736083984375] with confidence 0.98
 - black-pawn at [2326.05419921875, 1616.7515869140625, 2453.396484375, 1808.94677734375] with confidence 0.98
 - white-pawn at [988.3460693359375, 1564.974365234375, 1113.703369140625, 1755.7650146484375] with confidence 0.98
 - black-knight at [1784.1982421875, 815.6644287109375, 1900.5223388671875, 1025.607666015625] with confidence 0.98
 - white-rook at [1056.923095703125, 2

# ACABA AQUI

In [ ]:
# Faster R-CNN Chess Piece Detection with Best Model Saving
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

# Utility Functions
def load_categories(root_dir):
    """Load category names from annotations.json."""
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return {}
    with open(anns_file, 'r') as f:
        anns = json.load(f)
    return {c['id']: c['name'] for c in anns['categories']}

def get_test_image_paths(output_dir):
    """Get list of image paths in the test set."""
    test_image_dir = os.path.join(output_dir, 'test', 'images')
    if not os.path.exists(test_image_dir):
        print(f"Error: Test images directory {test_image_dir} not found")
        return []
    return [os.path.join(test_image_dir, img) for img in os.listdir(test_image_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

# Custom Dataset Class
class ChessDataset(Dataset):
    def __init__(self, root_dir, output_dir, partition='train', transforms=None):
        self.root_dir = root_dir
        self.output_dir = output_dir
        self.partition = partition
        self.transforms = transforms
        # Load annotations.json for category mapping
        anns_file = os.path.join(root_dir, 'annotations.json')
        if not os.path.exists(anns_file):
            raise FileNotFoundError(f"Annotations file {anns_file} not found")
        with open(anns_file, 'r') as f:
            self.anns = json.load(f)
        self.image_info = {img['id']: img for img in self.anns['images']}
        # Get image IDs for the partition
        if partition not in self.anns['splits']['chessred2k']:
            raise ValueError(f"Partition '{partition}' not found in annotations.json")
        split_ids = np.asarray(self.anns['splits']['chessred2k'][partition]['image_ids']).astype(int)
        self.image_ids = [img['id'] for img in self.anns['images'] if img['id'] in split_ids]
        # Use existing images and labels from yolo_chess_dataset
        image_dir = os.path.join(output_dir, partition, 'images')
        label_dir = os.path.join(output_dir, partition, 'labels')
        if not os.path.exists(image_dir) or not os.path.exists(label_dir):
            raise FileNotFoundError(f"Image or label directory missing for {partition}: {image_dir}, {label_dir}")
        self.image_paths = []
        self.label_paths = []
        images_found = 0
        labels_found = 0
        for image_id in self.image_ids:
            img_info = self.image_info[image_id]
            file_name = os.path.basename(img_info['path'])
            img_path = os.path.join(image_dir, file_name)
            label_path = os.path.join(label_dir, os.path.splitext(file_name)[0] + '.txt')
            if os.path.exists(img_path) and os.path.exists(label_path):
                self.image_paths.append(img_path)
                self.label_paths.append(label_path)
                images_found += 1
                labels_found += 1
            else:
                print(f"Warning: Missing image or label for {image_id}: {img_path}, {label_path}")
        print(f"Partition {partition}: Found {images_found} images and {labels_found} label files")
        if not self.image_paths:
            raise RuntimeError(f"No valid images or labels found for {partition} partition")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_path = self.label_paths[idx]
        img_info = next((info for info in self.image_info.values() if os.path.basename(info['path']) == os.path.basename(img_path)), None)
        if img_info is None:
            raise ValueError(f"No image info found for {img_path}")
        
        # Load image
        img = Image.open(img_path).convert('RGB')
        width, height = img_info['width'], img_info['height']
        
        # Read YOLO-style label file
        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    class_id, center_x, center_y, norm_w, norm_h = map(float, line.strip().split())
                    # Convert YOLO format to Faster R-CNN format (x_min, y_min, x_max, y_max)
                    x_min = (center_x - norm_w / 2) * width
                    y_min = (center_y - norm_h / 2) * height
                    x_max = (center_x + norm_w / 2) * width
                    y_max = (center_y + norm_h / 2) * height
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(class_id))
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([img_info['id']])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.tensor([])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': image_id,
            'area': area,
            'iscrowd': iscrowd
        }
        
        if self.transforms:
            img = self.transforms(img)
        
        return img, target

# Data Preparation
def get_transform(train=True):
    transforms = [torchvision.transforms.PILToTensor(), torchvision.transforms.ConvertImageDtype(torch.float32)]
    if train:
        transforms.append(torchvision.transforms.RandomHorizontalFlip(0.5))
    return torchvision.transforms.Compose(transforms)

# Main Execution
def main():
    # Configuration
    root_dir = ''  # Update with your dataset path containing annotations.json
    output_dir = 'yolo_chess_dataset'  # Use existing YOLO dataset
    num_epochs = 10
    batch_size = 8  # Updated to 16 as requested
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    conf_threshold = 0.7
    num_classes = len(load_categories(root_dir)) + 1  # +1 for background
    best_val_loss = float('inf')  # Track best validation loss
    best_model_path = os.path.join(output_dir, 'faster_rcnn_chess_best.pt')
    
    # Prepare datasets
    try:
        train_dataset = ChessDataset(root_dir, output_dir, partition='train', transforms=get_transform(train=True))
        val_dataset = ChessDataset(root_dir, output_dir, partition='val', transforms=get_transform(train=False))
        test_dataset = ChessDataset(root_dir, output_dir, partition='test', transforms=get_transform(train=False))
    except Exception as e:
        print(f"Error initializing datasets: {str(e)}")
        return
    
    if len(train_dataset) == 0:
        print("Error: No training images or labels found. Cannot proceed with training.")
        return
    if len(val_dataset) == 0:
        print("Warning: No validation images or labels found. Using training set for validation.")
        val_dataset = train_dataset
    
    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))
    
    # Verify image and label directories
    for split in ['train', 'val', 'test']:
        img_dir = os.path.join(output_dir, split, 'images')
        label_dir = os.path.join(output_dir, split, 'labels')
        img_count = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
        label_count = len(os.listdir(label_dir)) if os.path.exists(label_dir) else 0
        print(f"{split} directory: {img_count} images in {img_dir}, {label_count} labels in {label_dir}")
    
    # Load Faster R-CNN model with pretrained weights
    try:
        model = fasterrcnn_resnet50_fpn(weights='COCO_V1')
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
        model.to(device)
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        return
    
    # Optimizer and scheduler
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    
    # Training loop with detailed metrics and best model saving
    print("Starting training...")
    try:
        for epoch in range(num_epochs):
            model.train()
            train_losses = {'loss_classifier': 0.0, 'loss_box_reg': 0.0, 'loss_objectness': 0.0, 'loss_rpn_box_reg': 0.0, 'total': 0.0}
            batch_count = 0
            for images, targets in train_loader:
                images = [image.to(device) for image in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                optimizer.zero_grad()
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                train_losses['total'] += losses.item()
                train_losses['loss_classifier'] += loss_dict.get('loss_classifier', torch.tensor(0.0)).item()
                train_losses['loss_box_reg'] += loss_dict.get('loss_box_reg', torch.tensor(0.0)).item()
                train_losses['loss_objectness'] += loss_dict.get('loss_objectness', torch.tensor(0.0)).item()
                train_losses['loss_rpn_box_reg'] += loss_dict.get('loss_rpn_box_reg', torch.tensor(0.0)).item()
                batch_count += 1
                
                losses.backward()
                optimizer.step()
            
            # Validation
            model.eval()
            val_losses = {'loss_classifier': 0.0, 'loss_box_reg': 0.0, 'loss_objectness': 0.0, 'loss_rpn_box_reg': 0.0, 'total': 0.0}
            val_batch_count = 0
            with torch.no_grad():
                for images, targets in val_loader:
                    images = [image.to(device) for image in images]
                    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())
                    val_losses['total'] += losses.item()
                    val_losses['loss_classifier'] += loss_dict.get('loss_classifier', torch.tensor(0.0)).item()
                    val_losses['loss_box_reg'] += loss_dict.get('loss_box_reg', torch.tensor(0.0)).item()
                    val_losses['loss_objectness'] += loss_dict.get('loss_objectness', torch.tensor(0.0)).item()
                    val_losses['loss_rpn_box_reg'] += loss_dict.get('loss_rpn_box_reg', torch.tensor(0.0)).item()
                    val_batch_count += 1
            
            # Average losses
            avg_train_loss = train_losses['total'] / max(batch_count, 1)
            avg_val_loss = val_losses['total'] / max(val_batch_count, 1)
            
            # Save best model if validation loss improves
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), best_model_path)
                print(f"Saved best model with Val Loss: {avg_val_loss:.4f} at {best_model_path}")
            
            lr_scheduler.step()
            # Print detailed metrics
            print(f"Epoch {epoch+1}/{num_epochs}:")
            print(f"  Train - Total Loss: {avg_train_loss:.4f}, "
                  f"Classifier: {train_losses['loss_classifier']/max(batch_count, 1):.4f}, "
                  f"Box Reg: {train_losses['loss_box_reg']/max(batch_count, 1):.4f}, "
                  f"Objectness: {train_losses['loss_objectness']/max(batch_count, 1):.4f}, "
                  f"RPN Box Reg: {train_losses['loss_rpn_box_reg']/max(batch_count, 1):.4f}")
            print(f"  Val   - Total Loss: {avg_val_loss:.4f}, "
                  f"Classifier: {val_losses['loss_classifier']/max(val_batch_count, 1):.4f}, "
                  f"Box Reg: {val_losses['loss_box_reg']/max(val_batch_count, 1):.4f}, "
                  f"Objectness: {val_losses['loss_objectness']/max(val_batch_count, 1):.4f}, "
                  f"RPN Box Reg: {val_losses['loss_rpn_box_reg']/max(val_batch_count, 1):.4f}")
    except Exception as e:
        print(f"Training failed: {str(e)}")
        return
    
    # Save final model
    final_model_path = os.path.join(output_dir, 'faster_rcnn_chess_final.pt')
    torch.save(model.state_dict(), final_model_path)
    print(f"Saved final model to {final_model_path}")
    
    # Inference on test images
    categories = load_categories(root_dir)
    test_image_paths = get_test_image_paths(output_dir)[:3]  # Limit to 3 images
    model.eval()
    
    if not test_image_paths:
        print("No test images available for inference.")
        return
    
    for test_image_path in test_image_paths:
        if not os.path.exists(test_image_path):
            print(f"Error: Image {test_image_path} not found")
            continue
        
        print(f"Testing on image: {test_image_path}")
        img = Image.open(test_image_path).convert('RGB')
        img_tensor = F.to_tensor(img).to(device)
        
        with torch.no_grad():
            predictions = model([img_tensor])[0]
        
        # Load image for display
        img_cv = cv2.imread(test_image_path)
        if img_cv is None:
            print(f"Error: Could not load image {test_image_path}")
            continue
        img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        img_annotated = img_cv.copy()
        
        # Draw predictions
        for box, score, label in zip(predictions['boxes'], predictions['scores'], predictions['labels']):
            if score >= conf_threshold:
                x1, y1, x2, y2 = map(int, box)
                cls = label.item()
                label_text = f"{categories.get(cls, 'unknown')} {score:.2f}"
                cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img_annotated, label_text, (x1, max(y1 - 10, 10)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
        
        img_annotated_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
        
        # Side-by-side display
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        ax1.imshow(img_rgb)
        ax1.set_title("Original Image")
        ax1.axis('off')
        ax2.imshow(img_annotated_rgb)
        ax2.set_title(f"Predictions on {os.path.basename(test_image_path)}")
        ax2.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Save annotated image
        predictions_dir = os.path.join(output_dir, 'predictions')
        os.makedirs(predictions_dir, exist_ok=True)
        output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(test_image_path)}")
        cv2.imwrite(output_image_path, img_annotated)
        print(f"Saved annotated image to: {output_image_path}")
        
        # Print results
        print(f"Test image: {test_image_path}")
        print(f"Detected {len([s for s in predictions['scores'] if s >= conf_threshold])} objects:")
        for box, score, label in zip(predictions['boxes'], predictions['scores'], predictions['labels']):
            if score >= conf_threshold:
                cls = label.item()
                print(f" - {categories.get(cls, 'unknown')} at {box.tolist()} with confidence {score:.2f}")

if __name__ == "__main__":
    main()

Partition train: Found 1442 images and 1442 label files
Partition val: Found 330 images and 330 label files
Partition test: Found 306 images and 306 label files
train directory: 1442 images in yolo_chess_dataset\train\images, 1442 labels in yolo_chess_dataset\train\labels
val directory: 330 images in yolo_chess_dataset\val\images, 330 labels in yolo_chess_dataset\val\labels
test directory: 306 images in yolo_chess_dataset\test\images, 306 labels in yolo_chess_dataset\test\labels
Starting training...


KeyboardInterrupt: 

In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
import ultralytics
import random

# Check Ultralytics version
print(f"Ultralytics version: {ultralytics.__version__}")
if ultralytics.__version__ < '8.3.146':
    print("Warning: Ultralytics version is outdated. Please update with 'pip install -U ultralytics'")

# Set random seed for reproducibility
random.seed(42)

def create_yolo_annotations(root_dir, output_dir, partition='train'):
    """
    Convert annotations.json to YOLO format for a given partition (train/val/test).
    Creates .txt files with class_id and normalized bbox coordinates.
    Returns list of image paths for the partition.
    """
    # Load annotations
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return [], {}
    anns = json.load(open(anns_file))
    categories = {c['id']: c['name'] for c in anns['categories']}
    image_info = {img['id']: img for img in anns['images']}

    # Check if partition exists
    if partition not in anns['splits']['chessred2k']:
        print(f"Error: Partition '{partition}' not found in annotations.json")
        return [], categories

    # Get split IDs
    split_ids = np.asarray(anns['splits']['chessred2k'][partition]['image_ids']).astype(int)
    image_ids = [img['id'] for img in anns['images'] if img['id'] in split_ids]
    image_paths = []
    images_processed = 0
    images_skipped = 0

    # Create output directory for labels
    label_dir = os.path.join(output_dir, partition, 'labels')
    image_dir = os.path.join(output_dir, partition, 'images')
    os.makedirs(label_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    for image_id in image_ids:
        img_info = image_info[image_id]
        file_name = img_info['path']
        img_path = os.path.join(root_dir, file_name)
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            images_skipped += 1
            continue
        width, height = img_info['width'], img_info['height']

        # Copy image to output directory
        output_img_path = os.path.join(image_dir, os.path.basename(file_name))
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not load image {img_path}")
            images_skipped += 1
            continue
        cv2.imwrite(output_img_path, img)
        image_paths.append(os.path.join(partition, 'images', os.path.basename(file_name)))
        images_processed += 1

        # Create YOLO annotation file
        label_path = os.path.join(label_dir, os.path.splitext(os.path.basename(file_name))[0] + '.txt')
        with open(label_path, 'w') as f:
            for piece in anns['annotations']['pieces']:
                if piece['image_id'] == image_id and 'bbox' in piece:
                    x, y, w, h = piece['bbox']
                    class_id = piece['category_id']
                    # Normalize coordinates: center_x, center_y, width, height
                    center_x = (x + w / 2) / width
                    center_y = (y + h / 2) / height
                    norm_w = w / width
                    norm_h = h / height
                    # Ensure coordinates are within [0, 1]
                    if 0 <= center_x <= 1 and 0 <= center_y <= 1 and norm_w > 0 and norm_h > 0:
                        f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                    else:
                        print(f"Warning: Invalid bbox for image_id {image_id}: {piece['bbox']}")

    print(f"Processed {images_processed} images for {partition} split, skipped {images_skipped}")
    return image_paths, categories

def create_data_yaml(root_dir, output_dir, train_paths, val_paths, test_paths, categories):
    """
    Create data.yaml file for YOLOv11 training with absolute paths.
    """
    # Get absolute paths
    train_dir = os.path.abspath(os.path.join(output_dir, 'train', 'images'))
    val_dir = os.path.abspath(os.path.join(output_dir, 'val', 'images'))
    test_dir = os.path.abspath(os.path.join(output_dir, 'test', 'images'))

    yaml_content = f"""
train: {train_dir}
val: {val_dir}
test: {test_dir}
nc: {len(categories)}
names: {list(categories.values())}
"""
    yaml_path = os.path.join(output_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    # Print data.yaml contents for debugging
    print(f"data.yaml contents:\n{yaml_content}")
    return yaml_path

def main():
    # Configuration
    root_dir = ''  # Dataset path
    output_dir = 'yolo_chess_dataset'  # Output directory for YOLO format
    model_name = 'yolo11n.pt'  # Pretrained YOLOv11 model
    img_size = 640  # Training image size
    epochs = 50  # Number of training epochs
    batch_size = 16  # Batch size (adjust based on GPU memory)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Create YOLO annotations for train, val, test
    print("Converting annotations for training set...")
    train_paths, categories = create_yolo_annotations(root_dir, output_dir, 'train')
    print("Converting annotations for validation set...")
    val_paths, _ = create_yolo_annotations(root_dir, output_dir, 'val')
    print("Converting annotations for test set...")
    test_paths, _ = create_yolo_annotations(root_dir, output_dir, 'test')

    # Debug: Print available splits
    anns_file = os.path.join(root_dir, 'annotations.json')
    if os.path.exists(anns_file):
        anns = json.load(open(anns_file))
        print(f"Available splits in annotations.json: {list(anns['splits']['chessred2k'].keys())}")
    else:
        print(f"Error: Annotations file {anns_file} not found")
        return

    # Check if train or val splits are empty
    if not train_paths:
        print("Error: No training images found. Cannot proceed with training.")
        return
    if not val_paths:
        print("Warning: No validation images found. Using training set for validation.")
        val_paths = train_paths  # Fallback to train set for validation

    # Create data.yaml
    yaml_path = create_data_yaml(root_dir, output_dir, train_paths, val_paths, test_paths, categories)
    print(f"Created data.yaml at {yaml_path}")

    # Verify directories exist
    for split in ['train', 'val', 'test']:
        img_dir = os.path.join(output_dir, split, 'images')
        if os.path.exists(img_dir) and len(os.listdir(img_dir)) > 0:
            print(f"{split} images directory: {img_dir} contains {len(os.listdir(img_dir))} images")
        else:
            print(f"Warning: {split} images directory {img_dir} is empty or does not exist")

    # Load YOLOv11 model
    print(f"Loading pretrained model {model_name} on {device}")
    try:
        model = YOLO(model_name)
    except Exception as e:
        print(f"Failed to load model: {str(e)}")
        return

    # Train the model
    print("Starting training...")
    try:
        results = model.train(
            data=yaml_path,
            imgsz=img_size,
            epochs=epochs,
            batch=batch_size,
            name='chess_piece_detection',
            plots=True,
            device=device,
            patience=20,  # Early stopping after 20 epochs without improvement
            save=True,  # Save checkpoints
            save_period=10  # Save every 10 epochs
        )
    except Exception as e:
        print(f"Training failed: {str(e)}")
        return

    # Test the model on a sample test image
    print("Testing model on a sample image...")
    if test_paths:
        test_image_path = os.path.join(output_dir, test_paths[0])
        if not os.path.exists(test_image_path):
            print(f"Error: Test image {test_image_path} not found")
            return
        results = model.predict(test_image_path, save=False, conf=0.5)

        # Load and display the test image with predictions
        img = cv2.imread(test_image_path)
        if img is None:
            print(f"Error: Could not load test image {test_image_path}")
            return
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Draw bounding boxes and labels
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()
                cls = int(box.cls.item())
                label = f"{categories[cls]} {conf:.2f}"
                # Draw rectangle (BGR for OpenCV, red)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                # Add label
                cv2.putText(img, label, (x1, max(y1 - 10, 10)), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        # Display the image
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"YOLOv11 Predictions on {os.path.basename(test_image_path)}")
        plt.show()

        # Print test results
        print(f"Test image: {test_image_path}")
        print(f"Detected {len(result.boxes)} objects:")
        for box in result.boxes:
            cls = int(box.cls.item())
            conf = box.conf.item()
            print(f" - {categories[cls]} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

    else:
        print("No test images available for inference.")

if __name__ == "__main__":
    main()

Ultralytics version: 8.3.146
Converting annotations for training set...
Processed 1442 images for train split, skipped 0
Converting annotations for validation set...
Processed 330 images for val split, skipped 0
Converting annotations for test set...
Processed 306 images for test split, skipped 0
Available splits in annotations.json: ['train', 'val', 'test']
data.yaml contents:

train: d:\VCOM\yolo_chess_dataset\train\images
val: d:\VCOM\yolo_chess_dataset\val\images
test: d:\VCOM\yolo_chess_dataset\test\images
nc: 13
names: ['white-pawn', 'white-rook', 'white-knight', 'white-bishop', 'white-queen', 'white-king', 'black-pawn', 'black-rook', 'black-knight', 'black-bishop', 'black-queen', 'black-king', 'empty']

Created data.yaml at yolo_chess_dataset\data.yaml
train images directory: yolo_chess_dataset\train\images contains 1442 images
val images directory: yolo_chess_dataset\val\images contains 330 images
test images directory: yolo_chess_dataset\test\images contains 306 images
Loading

100%|██████████| 5.35M/5.35M [00:02<00:00, 2.10MB/s]


Starting training...
New https://pypi.org/project/ultralytics/8.3.152 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.146  Python-3.11.5 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_chess_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=chess_pi

train: Scanning D:\VCOM\yolo_chess_dataset\train\labels... 1442 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1442/1442 [00:04<00:00, 355.76it/s]

train: New cache created: D:\VCOM\yolo_chess_dataset\train\labels.cache


...


50 epochs completed in 0.407 hours.
Optimizer stripped from runs\detect\chess_piece_detection\weights\last.pt, 5.5MB
Optimizer stripped from runs\detect\chess_piece_detection\weights\best.pt, 5.5MB

Validating runs\detect\chess_piece_detection\weights\best.pt...
Ultralytics 8.3.146  Python-3.11.5 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
YOLO11n summary (fused): 100 layers, 2,584,687 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:05<00:00,  2.18it/s]


                   all        330       6132      0.993      0.992      0.995      0.871
            white-pawn        330       1625      0.999      0.995      0.995      0.848
            white-rook        281        447      0.996          1      0.995      0.848
          white-knight        220        274          1      0.989      0.995      0.866
          white-bishop        235        335          1      0.991      0.995      0.864
           white-queen        126        126      0.992          1      0.995      0.915
            white-king        330        330      0.997      0.998      0.995      0.912
            black-pawn        330       1511      0.998      0.997      0.995      0.856
            black-rook        280        471      0.998      0.993      0.995      0.883
          black-knight        125        178      0.983      0.972      0.994      0.819
          black-bishop        223        380      0.998      0.979      0.995      0.858
           black-quee

<Figure size 640x480 with 1 Axes>

Test image: yolo_chess_dataset\test\images\G000_IMG000.jpg
Detected 32 objects:
 - white-rook at [2410.435546875, 1391.2392578125, 2556.31640625, 1585.6058349609375] with confidence 0.98
 - black-rook at [524.1851196289062, 963.0350341796875, 653.1126098632812, 1145.84228515625] with confidence 0.98
 - white-queen at [1670.716796875, 1653.9818115234375, 1824.3785400390625, 1938.3594970703125] with confidence 0.97
 - black-pawn at [1477.205322265625, 801.9133911132812, 1569.2320556640625, 952.1345825195312] with confidence 0.97
 - white-king at [1856.922607421875, 1540.57421875, 2021.90234375, 1849.4783935546875] with confidence 0.97
 - black-knight at [1538.9376220703125, 599.3533325195312, 1642.3076171875, 793.9320068359375] with confidence 0.97
 - black-queen at [1055.0556640625, 711.1854858398438, 1183.136474609375, 966.1668701171875] with confidence 0.97
 - black-rook at [1690.124755859375, 567.7242431640625, 1796.2041015625, 737.964599609375] with confidence 0.97
 - white-knight a

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

def load_categories(root_dir):
    """
    Load category names from annotations.json.
    Returns a dictionary mapping category_id to name.
    """
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return {}
    with open(anns_file, 'r') as f:
        anns = json.load(f)
    return {c['id']: c['name'] for c in anns['categories']}

def get_test_image_paths(output_dir):
    """
    Get list of image paths in the test set.
    """
    test_image_dir = os.path.join(output_dir, 'test', 'images')
    if not os.path.exists(test_image_dir):
        print(f"Error: Test images directory {test_image_dir} not found")
        return []
    return [os.path.join(test_image_dir, img) for img in os.listdir(test_image_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

def test_on_image(model, image_path, categories, output_dir, conf_threshold=0.5):
    """
    Run inference on a single image, display results, and save annotated image.
    """
    if not os.path.exists(image_path):
        print(f"Error: Image {image_path} not found")
        return

    # Run inference
    results = model.predict(image_path, save=False, conf=conf_threshold)

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image {image_path}")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # For display
    img_annotated = img.copy()  # For saving (BGR)

    # Draw bounding boxes and labels
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            # Draw rectangle (BGR for OpenCV, red)
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            # Add label
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    # Display the image
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"YOLOv11 Predictions on {os.path.basename(image_path)}")
    plt.show()

    # Save annotated image
    predictions_dir = os.path.join(output_dir, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)
    output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(image_path)}")
    cv2.imwrite(output_image_path, img_annotated)
    print(f"Saved annotated image to: {output_image_path}")

    # Print results
    print(f"Test image: {image_path}")
    print(f"Detected {len(result.boxes)} objects:")
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = box.conf.item()
        print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

def main():
    # Configuration
    root_dir = ''  # Dataset path
    output_dir = 'yolo_chess_dataset'  # YOLO dataset path
    model_path = 'runs/detect/chess_piece_detection10/weights/best.pt'  # Path to trained model weights
    conf_threshold = 0.5  # Confidence threshold for predictions

    # Load categories
    categories = load_categories(root_dir)
    if not categories:
        return

    # Load test image paths
    test_image_paths = get_test_image_paths(output_dir)
    if not test_image_paths:
        print("No test images available for inference.")
        return

    # Select a test image (first one by default)
    test_image_path = test_image_paths[0]  # Change index or specify a specific image path
    # Example: test_image_path = r'C:\Users\migue\Desktop\Feup\FEUP\MEIC\ano1_semestre2\VC\TP9\yolo_chess_dataset\test\images\G000_IMG000.jpg'

    # Load YOLOv11 model
    print(f"Loading trained model from {model_path}")
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f"Failed to load model: {str(e)}")
        return

    # Test on the selected image
    print(f"Testing on image: {test_image_path}")
    test_on_image(model, test_image_path, categories, output_dir, conf_threshold)

if __name__ == "__main__":
    main()

Loading trained model from runs/detect/chess_piece_detection10/weights/best.pt
Failed to load model: [Errno 2] No such file or directory: 'runs\\detect\\chess_piece_detection10\\weights\\best.pt'
